In [ ]:
library(tidyverse)

# Load SOD1
sod1 <- read_tsv('../1.SOD1/sod1_pathogenic_carrier_status_by_id.txt', comment = '#')
cat('SOD1:', nrow(sod1), '\n')
table(sod1$Carrier_Status)

In [ ]:
# Load C9orf72
c9 <- read_tsv('../2.C9orf72/c9orf72_genotypes_results.tsv')
cat('C9orf72:', nrow(c9), '\n')

# Parse repeat counts - flag if any allele > 100
c9 <- c9 %>%
  mutate(
    allele1 = as.numeric(str_extract(c9orf72_REPCN, '^[0-9]+')),
    allele2 = as.numeric(str_extract(c9orf72_REPCN, '[0-9]+$')),
    c9_expanded = (allele1 > 100 | allele2 > 100)
  ) %>%
  rename(IID = eid)

cat('C9 expanded carriers:', sum(c9$c9_expanded, na.rm = TRUE), '\n')

In [ ]:
# Load Other genes analysis/06-UKB_analyisis/00-Data-prepration/3.Genotype/3.Other/eid_carrier_any_als_pathogenic_153_mutations.csv
other_df <- read_csv('../3.Other/SOD1_other_genes_mutation_summary_1027.csv') # due to keep the same name as the original submission, we use the same file as the original submission

colnames(other_df)[1] <- 'IID'

In [ ]:
library(dplyr)
library(readr)
library(stringr)

make_carrier01 <- function(df, gene) {
  df %>%
    transmute(
      IID = as.character(IID),
      !!paste0("carrier_", gene) := as.integer(str_detect(tolower(Carrier_Status), "carrier") &
                                              !str_detect(tolower(Carrier_Status), "non"))
      # safer alternative (recommended):
      # !!paste0("carrier_", gene) := as.integer(tolower(Carrier_Status) == "carrier")
    )
}

df_ANXA11 <- read_tsv("../3.Other/gene_extraction_results/UKB_anxa11_pathogenic_carrier_status_by_id.txt",
                      comment = "#", show_col_types = FALSE)
df_OPTN   <- read_tsv("../3.Other/gene_extraction_results/UKB_optn_pathogenic_carrier_status_by_id.txt",
                      comment = "#", show_col_types = FALSE)
df_VCP    <- read_tsv("../3.Other/gene_extraction_results/UKB_vcp_pathogenic_carrier_status_by_id.txt",
                      comment = "#", show_col_types = FALSE)

a <- make_carrier01(df_ANXA11, "ANXA11")
o <- make_carrier01(df_OPTN,   "OPTN")
v <- make_carrier01(df_VCP,    "VCP")

df_any_als <- a %>%
  full_join(o, by = "IID") %>%
  full_join(v, by = "IID") %>%
  mutate(
    carrier_any_als = as.integer(
      coalesce(carrier_ANXA11, 0L) == 1L |
      coalesce(carrier_OPTN,   0L) == 1L |
      coalesce(carrier_VCP,    0L) == 1L
    )
  )

  df_any_als$IID <- as.double(df_any_als$IID)

In [ ]:
# Build master dataframe
df <- sod1 %>%
  select(IID, sod1_status = Carrier_Status) %>%
  left_join(c9 %>% select(IID, c9_expanded), by = 'IID') %>% 
  left_join(other_df %>% select(IID, other_status = AnyMutation), by = 'IID') %>%
  left_join(df_any_als %>% select(IID, other_status_als = carrier_any_als), by = 'IID')

cat('Merged:', nrow(df), '\n')

In [ ]:
# Assign genotype group
# Priority: C9orf72 > SOD1 nonA4V > Other Genotype > None identified
df <- df %>%
  mutate(
    Genotype_Group = case_when(
      c9_expanded == TRUE ~ 'C9orf72',
      sod1_status == 'Carrier' ~ 'SOD1 nonA4V',
      other_status == 1 ~ 'Other Genotype',
      TRUE ~ 'None identified'
    )
  )

table(df$Genotype_Group)

In [ ]:
# Save
df %>%
  select(IID, Genotype_Group,other_status_als) %>%
  write_csv(here::here("data", "analysis_data", "ukb", "visit_info", "UKB_geno.csv"))
